# Forest cover types (581,012 rows)

The [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype) describes 30 m × 30 m cells in the Roosevelt National Forest, Colorado. Each cell has 10 cartographic measurements (elevation, slope, distances to water, roads and fire points, hillshade) and 44 binary indicators (4 wilderness areas, 40 soil types). The target is the dominant tree species.

With more than half a million rows, this example exercises Atlas's *anchor + projection* path:

1. UMAP runs in the browser on 100,000 anchor rows (25,000 without WebGPU).
2. The kernel places the remaining rows next to their nearest anchors, using numpy only.

In [ ]:
import gzip
import urllib.request
from pathlib import Path

import numpy as np

CACHE = Path.home() / ".cache" / "positron-atlas" / "datasets"


def download(url: str, name: str) -> Path:
    """Download a file once and cache it."""
    path = CACHE / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {url} ...")
        tmp = path.with_suffix(path.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    return path


def read_idx(path: Path) -> np.ndarray:
    """Read an IDX file (the MNIST format): images as (n, 784) uint8, labels as (n,)."""
    with gzip.open(path, "rb") as f:
        data = f.read()
    ndim = data[3]
    shape = tuple(int.from_bytes(data[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim))
    arr = np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim)
    return arr.reshape(shape[0], -1) if ndim > 1 else arr

In [ ]:
import polars as pl

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"
path = download(URL, "covtype.data.gz")

QUANTITATIVE = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
WILDERNESS = ["Rawah", "Neota", "Comanche Peak", "Cache la Poudre"]
COVER = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]

raw = np.loadtxt(path, delimiter=",", dtype=np.int32)  # (581012, 55), takes a few seconds
raw.shape

Keep the 10 measurements as numeric features. Turn the one-hot indicators into categorical columns, so they are available for coloring and not used as features.

In [ ]:
covtype = pl.DataFrame({name: raw[:, i].astype(np.float32) for i, name in enumerate(QUANTITATIVE)}).with_columns(
    pl.Series("cover_type", [COVER[i - 1] for i in raw[:, 54]], dtype=pl.Enum(COVER)),
    pl.Series("wilderness", [WILDERNESS[i] for i in raw[:, 10:14].argmax(axis=1)], dtype=pl.Enum(WILDERNESS)),
    pl.Series("soil_type", [f"soil {i + 1}" for i in raw[:, 14:54].argmax(axis=1)]),
)
covtype.shape

## Open it in Atlas

1. In the **Variables** pane, double-click **`covtype`** to open it in the Data Explorer.
2. Click the **Open in Atlas (UMAP)** button (scatter icon) in the Data Explorer toolbar.

Or run **Atlas: Open DataFrame…** from the Command Palette and pick `covtype`.

## Things to try

- **Color by `cover_type`.** This is the default. Then try `wilderness` and `elevation`: elevation explains much of the species structure.
- **Watch the footer.** It shows `UMAP on 100,000 anchors`, then the remaining rows are placed in the kernel (a few seconds) and fill in progressively.
- **Lower *Sample rows*** to 10 % and click **Apply** for a quick, fully exact UMAP on 58,000 rows. No placement is needed then.
- **Add coordinates to DataFrame.** This writes `umap_x`/`umap_y` back into `covtype`. Because polars DataFrames are immutable, the variable is rebound to a new DataFrame. Then try, for example:

  ```python
  covtype.group_by("cover_type").agg(pl.col("umap_x", "umap_y").mean())
  ```